# Preprocessing Notebook

A summary of preprocessing steps for machine learning.

## 📚 1. Imports

All the libraries we may use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, LabelEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance
from sklearn.decomposition import PCA, KernelPCA

# Imbalanced learning
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

# Models for examples
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

## ⚠️ 2. Train-Test Split (The Golden Rule)

**Why:** This is the **most important step** and *must* be done before any other preprocessing.

All preprocessing steps (like imputation or scaling) must be **`fit` on the `X_train` data only**. This prevents "data leakage," where information from the test set (e.g., its mean) leaks into the training process, giving you an unrealistically good score.

**When:** Always do this first.
**Pro-Tip:** Use `stratify=y` for classification to ensure both your train and test sets have the same proportion of classes as the original dataset. This is critical for imbalanced data.

In [ ]:
# Assume 'df' is your full dataset and 'target' is the name of your y-variable
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,    # 20% for testing
    random_state=42,  # For reproducibility
    stratify=y        # For classification
)

## 🧹 3. Basic Cleaning

Handle duplicates and missing values.

### 3a. Drop Duplicates

**Why:** Duplicates can bias the model. This step is one of the few you do *before* the train-test split.

In [ ]:
df = df.drop_duplicates()

### 3b. Handle Missing Values (NaN)

First, check how much data is missing. Decisions are made *using the train set only*.

In [ ]:
# missing_train = X_train.isnull().sum()
print(missing_train / len(X_train))

**Option 1: Drop Columns**

**Why:** If a column is missing a huge amount of data (e.g., > 40%), it's often better to drop it entirely as it contains little information.

In [ ]:
# 1. Find columns to drop based on X_train
missing_percent = X_train.isnull().sum() / len(X_train)
features_to_drop = missing_percent[missing_percent > 0.3].index

# 2. Drop them from BOTH train and test
X_train = X_train.drop(columns=features_to_drop)
X_test = X_test.drop(columns=features_to_drop)

**Option 2: Imputation (Filling Values)**

**Why:** This is preferred if data is missing < 40% of the time. It saves the feature. We use `SimpleImputer`.

* **Numerical:** Use `strategy='median'` (good against outliers) or `strategy='mean'`. 
* **Categorical:** Use `strategy='most_frequent'` (fills with the most common category) or `strategy='constant'` (fills with a new category like "Missing").

In [ ]:
# This step is best done inside a Pipeline (see Step 8)

num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

# 1. Fit on TRAIN data
num_imputer.fit(X_train[numerical_cols])

# 2. Transform BOTH train and test
X_train[numerical_cols] = num_imputer.transform(X_train[numerical_cols])
X_test[numerical_cols] = num_imputer.transform(X_test[numerical_cols])

## 🎯 4. Handling Outliers

**Why:** Extreme values (outliers) can skew the results of models like Linear/Logistic Regression and `StandardScaler`.



### Capping (Winsorization)

**What:** This is the most common method. Instead of *removing* outliers, we "cap" them at a maximum/minimum value (e.g., the 1st and 99th percentiles, or using the IQR rule).

**Why:** It reduces the outlier's influence while keeping the rest of the row's data.

In [ ]:
def iqr_caps(frame, whisker=1.5):
    """Return per-column lower/upper caps from IQR, computed on `frame`."""
    q1 = frame.quantile(0.25, numeric_only=True)
    q3 = frame.quantile(0.75, numeric_only=True)
    iqr = q3 - q1
    lower = q1 - whisker * iqr
    upper = q3 + whisker * iqr
    return lower, upper

def winsorize(frame, lower, upper):
    """Cap values using provided per-column lower/upper Series."""
    capped = frame.copy()
    for c in frame.columns:
        if np.issubdtype(frame[c].dtype, np.number):
            if c in lower.index:
                capped[c] = np.clip(frame[c], lower[c], upper[c])
    return capped

# 1. Compute caps on TRAIN ONLY
lower_cap, upper_cap = iqr_caps(X_train, whisker=1.5)

# 2. Apply to train & test
X_train_wins = winsorize(X_train, lower_cap, upper_cap)
X_test_wins  = winsorize(X_test,  lower_cap, upper_cap)

## 🔡 5. Encoding Categorical Features

**Why:** Models only understand numbers, so we must convert string categories (like "Red", "Blue") into numerical values.

First, identify your categorical and numerical columns:

In [ ]:
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns
numerical_cols = X_train.select_dtypes(include=['float64', 'int64']).columns

### 5a. One-Hot Encoder (OHE)

**What:** Converts one column (e.g., `Color`) into *multiple* columns (`Color_Red`, `Color_Blue`, etc.), with a 0 or 1.

**When:** Use this for **nominal** (unordered) categories, like `Color` or `Country`. This is the safest and most common method.

`handle_unknown="ignore"` is important: it prevents errors if the test set has a category that wasn't in the train set.

In [ ]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

### 5b. Ordinal Encoder

**What:** Converts one column of categories into *one* column of numbers (e.g., `Small=0`, `Medium=1`, `Large=2`).

**When:** Use this *only* for **ordinal** (ordered) categories, like `Education_Level` or `Quality_Rating`.

In [ ]:
# 1. Define the order of your categories
quality_categories = ['Poor', 'Good', 'Excellent']

# 2. Create the encoder
ord_encoder = OrdinalEncoder(categories=[quality_categories])

# 3. Fit on train, transform both
X_train[['quality']] = ord_encoder.fit_transform(X_train[['quality']])
X_test[['quality']] = ord_encoder.transform(X_test[['quality']])

### 5c. Label Encoder (for y only!)

**What:** Just like `OrdinalEncoder`, it converts categories to `0, 1, 2...`

**When:** ⚠️ **Critical:** Only use this for your **target variable (`y`)**. Never use it on your features (`X`) unless they are *truly* ordinal, as it will make your model think `Country_USA` (e.g., `2`) is greater than `Country_Canada` (e.g., `1`).

In [ ]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)

## 📈 6. Scaling & Transforming Numerical Features

**Why:** Models like `LogisticRegression`, `KNN`, `SVM`, and `PCA` are sensitive to scale. A feature from 0-1,000,000 will be treated as more important than a feature from 0-1. Scaling puts all features on a level playing field. (Tree-based models like `RandomForest` don't need scaling).

### 6a. StandardScaler

**What:** Scales data to have a **mean of 0** and **standard deviation of 1** (a "z-score").

**When:** The default, go-to scaler. It's great if your data is somewhat normally distributed. It is sensitive to outliers.

In [ ]:
scaler = StandardScaler()

### 6b. MinMaxScaler

**What:** Scales data to a fixed range, usually **[0, 1]**.

**When:** Good for algorithms that require data in a specific range, like Neural Networks or KNN. It is *very* sensitive to outliers.

In [ ]:
scaler = MinMaxScaler()

### 6c. RobustScaler

**What:** Scales data using the **Interquartile Range (IQR)**. It removes the median and scales by the IQR.

**When:** The best choice if your data **has outliers**. It is *robust* to their influence.

In [ ]:
scaler = RobustScaler()

### 6d. Log Transform (for Skewed Data)

**What:** Applies a log function to a feature.

**When:** Use this if a feature is **highly skewed** (e.g., `income`, `population`). It pulls in the long tail and can make the distribution more normal. Use `np.log1p` which handles `0` values.

In [ ]:
def log_function(x):
    return np.log1p(x)  # np.log1p(x) is log(x + 1)

# Create the transformer
log_transformer = FunctionTransformer(log_function)

## ⚖️ 7. Handling Imbalanced Datasets

**Why:** If your target `y` is 99% "No" and 1% "Yes", your model will learn to just always say "No". These techniques help the model pay attention to the rare class. **Important:** Only apply these to the **training data**.

### 7a. Class Weights

**What:** Tell the model to penalize mistakes on the minority class more heavily.

**When:** This is the easiest and cleanest method. Just set `class_weight="balanced"` in the model.

In [ ]:
rf_model = RandomForestClassifier(class_weight='balanced', random_state=42)

### 7b. Undersampling

**What:** Randomly removes examples from the *majority* class to balance the dataset.

**When:** Use this if you have a *huge* dataset and can afford to throw away data. Be careful, as you lose information.

In [ ]:
undersampler = RandomUnderSampler(random_state=42)
X_train_res, y_train_res = undersampler.fit_resample(X_train, y_train)

### 7c. Oversampling (SMOTE)

**What:** **S**ynthetic **M**inority **O**ver-sampling **TE**chnique. It *creates* new, synthetic examples of the minority class.

**When:** This is generally preferred over undersampling as it doesn't throw away data. It's the most common resampling method.

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

### 7d. Threshold Moving

**What:** After training, you change the decision threshold. Instead of predicting "Yes" if `prob > 0.5`, you might set it to `prob > 0.2` to find more of the rare class.

**When:** Used after prediction to tune for business goals (e.g., to improve Recall).

## ⛓️ 8. Building the Pipeline

**Why:** This is the best practice. A `Pipeline` chains all your steps (e.g., Imputer -> Scaler). A `ColumnTransformer` applies different pipelines to different columns (e.g., a numerical pipeline for num_cols, a categorical one for cat_cols).

This prevents data leakage and makes your code clean and reproducible.

In [ ]:
# 1. Define the numerical pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())  # Or RobustScaler(), MinMaxScaler()
])

In [ ]:
# 2. Define the categorical pipeline
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [ ]:
# 3. Use ColumnTransformer to apply them to the right columns

# Assume 'numerical_cols' and 'categorical_cols' are lists of column names

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='passthrough' # Keeps columns not specified
)

### Putting it all in one final pipeline with a model:

In [ ]:
# 4. Create the full pipeline
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

# 5. Train the *entire* pipeline on raw training data
final_pipeline.fit(X_train, y_train)

# 6. Score on raw test data (it auto-transforms it)
score = final_pipeline.score(X_test, y_test)

## 🔎 9. Feature Selection

**Why:** To reduce the number of features. This can reduce overfitting, speed up training, and make the model easier to interpret. This is different from PCA, as we *keep* original features, not create new ones.

### 9a. Correlation Heatmap

**What:** A fast check for **redundant** features.

**When:** If two features have a very high correlation (e.g., > 0.8 or < -0.8), they are telling the model the same thing. You can drop one of them.

In [ ]:
# 1. Calculate correlation on TRAIN data
corr = X_train[numerical_cols].corr()

# 2. Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap="seismic", annot=True, fmt=".2f");

### 9b. Using a Pairplot

**What (To Look For):**
A pairplot gives you a comprehensive first look at your data. You use it to **diagnose** your features and their interactions.

1.  **Check the Diagonal (Distributions):**
    * **What it is:** A histogram or density plot for each feature.
    * **What to look for:**
        * **Shape:** Is it **normally distributed** (a bell curve)? Or is it **skewed** (lopsided)? Is it **bimodal** (two peaks)?
        * **Outliers:** Are there isolated bars or bumps far from the main group?

2.  **Check the Off-Diagonal (Relationships):**
    * **What it is:** Scatter plots showing every feature against every other feature.
    * **What to look for:**
        * **Correlation:** Do the points form a line (linear correlation)? A curve (non-linear relationship)? Or just a random cloud (no relationship)?
        * **Clusters:** Do the data points naturally clump into separate groups? 
        * **Outliers:** Are there single points far away from the main clusters in the scatter plot?

**When (What To Do):**
Based on what you see, you decide on your next data cleaning and modeling steps.

* **If you see a skewed distribution...**
    * **Action:** You might need to **transform** that feature (e.g., apply a log transform) to make it more like a bell curve, which helps many models perform better.
* **If you see a strong linear relationship...**
    * **Action:** This confirms a high correlation (like you saw in the heatmap). You can confidently **drop one of the features** to reduce redundancy.
* **If you see a non-linear (curved) relationship...**
    * **Action:** A simple linear model won't capture this. You might need to **create new features** (e.g., polynomial features) or use a more complex model (like a Random Forest) that can handle curves.
* **If you see clear clusters...**
    * **Action:** This is a major insight! It might suggest you should **build separate models** for each cluster or use the cluster itself as a new feature.
* **If you spot outliers...**
    * **Action:** You need to **investigate** them. Are they data entry errors (fix or remove)? Or are they real, important events (keep them)?

In [ ]:
# 1. Create the pairplot, coloring by the 'species' column
sns.pairplot(X_train[numerical_cols], hue='species')

# 2. Show the plot
plt.show()

### 9c. Permutation Importance

**What:** A powerful method to see which features are *actually* important to a model.

**How:** 1. Train a model. 2. Get its score. 3. *Shuffle* one feature (e.g., `age`) and get the score again. If the score drops a lot, `age` was very important. 

**When:** After you have a trained model, to verify what it learned.

In [ ]:
# 1. First, you need a trained model on your processed data
model = LogisticRegression(max_iter=3000).fit(X_train_processed, y_train)

# 2. Run permutation importance (best on a validation set)
permutation_score = permutation_importance(model, X_train_processed, y_train, n_repeats=10, random_state=42)

# 3. View the results
importance_df = pd.DataFrame(
    np.vstack((X_train_processed.columns, permutation_score.importances_mean)).T, 
    columns=['feature','score decrease']
)

importance_df.sort_values(by="score decrease", ascending = False)

## ⚡ 10. Dimensionality Reduction (PCA)

**Why:** To reduce many features (e.g., 100s) into a few new, synthetic features (called "components"). This is different from feature *selection* because the new components are combinations of the old ones.

**When:** Use this if you have a huge number of features and your model is slow or overfitting. **Data must be scaled first!**

### 10a. PCA (Principal Component Analysis)

**What:** A linear method that finds new axes (components) that capture the most *variance* in the data.

**When:** The standard, go-to method. You can set `n_components=0.95` to tell it to "keep however many components are needed to explain 95% of the variance."

In [ ]:
# 1. Must use SCALED data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled = scaler.transform(X_test[numerical_cols])

# 2. Fit PCA on scaled train data
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_scaled)

# 3. Transform test data
X_test_pca = pca.transform(X_test_scaled)

# 4. Check how many components were kept
print(f"Original features: {X_train_scaled.shape[1]}")
print(f"Reduced features: {X_train_pca.shape[1]}")

In [ ]:
# To find the "elbow" point, you can plot the explained variance
pca_full = PCA().fit(X_train_scaled)
plt.plot(np.cumsum(pca_full.explained_variance_ratio_))
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Explained Variance Plot')
plt.show()

### 10b. Kernel PCA (kPCA)

**What:** An advanced version of PCA that can find non-linear relationships.

**When:** Use this if your data has complex, non-linear structures (e.g., shapes like a spiral or two moons) that linear PCA can't capture.

In [ ]:
kpca = KernelPCA(n_components=2, kernel='rbf', gamma=0.1)\n",
X_kpca = kpca.fit_transform(X_train_scaled)